In [1]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import numpy as np

from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms

In [2]:
import os

import shutil
torch.manual_seed(0)

In [3]:


# 1. 전처리 정의 (VGG16 기준)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. 전체 데이터셋 불러오기
full_dataset = datasets.ImageFolder(root='archive/dataset/dataset', transform=transform)

# 3. 비율 설정 (예: 훈련 80%, 검증 20%)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

# 4. 데이터 분할
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 5. 로더 생성
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [4]:
import torch.nn as nn
from torchvision import models
import torch.optim as optim

# 모델 불러오기 및 수정
model = models.vgg16(pretrained=True)

# 가중치 고정 (특징 추출부)
for param in model.features.parameters():
    param.requires_grad = False

# 분류기 수정 (Animals 151 맞춤)
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, 151)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 손실함수 및 최적화 도구
crossloss = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)


/home/hi/miniconda3/envs/torch_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hi/miniconda3/envs/torch_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
train_losses, train_accs = [], []
val_losses, val_accs = [], []

for epoch in range(20):
    # --- [TRAIN] ---
    model.train()
    t_loss, t_acc = 0, 0
    for i, j in train_loader:
        i, j = i.to(device), j.to(device)
        
        optimizer.zero_grad()
        z = model(i)
        loss = crossloss(z, j)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item()
        t_acc += (z.argmax(1) == j).sum().item()
    
    # --- [VALIDATION] ---
    model.eval()
    v_loss, v_acc = 0, 0
    with torch.no_grad():
        for i, j in val_loader:
            i, j = i.to(device), j.to(device)
            z = model(i)
            v_loss += crossloss(z, j).item()
            v_acc += (z.argmax(1) == j).sum().item()
            
    # 에폭 결과 계산 및 저장
    train_losses.append(t_loss / len(train_loader))
    train_accs.append(t_acc / len(train_dataset))
    val_losses.append(v_loss / len(val_loader))
    val_accs.append(v_acc / len(val_dataset))
    
    print(f"Epoch [{epoch+1}/20] | Train Acc: {train_accs[-1]:.4f} | Val Acc: {val_accs[-1]:.4f}")


Epoch [1/20] | Train Acc: 0.4898 | Val Acc: 0.7823
Epoch [2/20] | Train Acc: 0.8702 | Val Acc: 0.8341
Epoch [3/20] | Train Acc: 0.9514 | Val Acc: 0.8317
Epoch [4/20] | Train Acc: 0.9779 | Val Acc: 0.8549
Epoch [5/20] | Train Acc: 0.9910 | Val Acc: 0.8461
Epoch [6/20] | Train Acc: 0.9912 | Val Acc: 0.8389
Epoch [7/20] | Train Acc: 0.9906 | Val Acc: 0.8469
Epoch [8/20] | Train Acc: 0.9944 | Val Acc: 0.8628
Epoch [9/20] | Train Acc: 0.9942 | Val Acc: 0.8533
Epoch [10/20] | Train Acc: 0.9942 | Val Acc: 0.8469
Epoch [11/20] | Train Acc: 0.9952 | Val Acc: 0.8573
Epoch [12/20] | Train Acc: 0.9964 | Val Acc: 0.8485
Epoch [13/20] | Train Acc: 0.9936 | Val Acc: 0.8413
Epoch [14/20] | Train Acc: 0.9912 | Val Acc: 0.8373
Epoch [15/20] | Train Acc: 0.9916 | Val Acc: 0.8254
Epoch [16/20] | Train Acc: 0.9892 | Val Acc: 0.8477
Epoch [17/20] | Train Acc: 0.9940 | Val Acc: 0.8501
Epoch [18/20] | Train Acc: 0.9888 | Val Acc: 0.8445
Epoch [19/20] | Train Acc: 0.9902 | Val Acc: 0.8246
Epoch [20/20] | Train

In [6]:
model.eval()
test_acc = 0
with torch.no_grad():
    for i, j in val_loader: # 혹은 test_loader
        i, j = i.to(device), j.to(device)
        z = model(i)
        test_acc += (z.argmax(1) == j).sum().item()

print(f"\n최종 정확도: {(test_acc / len(val_dataset)) * 100:.2f}%")



최종 정확도: 83.49%


In [7]:
import numpy as np


np.save('loss1.npy', np.array(val_losses))
np.save('acc1.npy', np.array(val_accs))
print('저장함')

저장함
